In [1]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
from pathlib import Path

In [2]:
@dataclass
class Config:
    n_rows: int = 5000
    random_state: int = 42
    artifacts_dir: str = "artifacts"

CFG = Config()
Path(CFG.artifacts_dir).mkdir(parents=True, exist_ok=True)

In [3]:
FEATURE_COLUMNS = [
    "crime_density",
    "crime_trend",
    "crime_type_violence_ratio",
    "transit_lines_nearby",
    "transit_frequency",
    "street_connectivity",
    "avg_lighting_quality",
    "crowd_density",
    "weather_visibility",
    "hour_of_day",
    "day_of_week",
    "is_holiday",
    "route_length_km",
    "num_intersections",
]
TARGET_COLUMN = "risk"

In [4]:
# Cell 4: Synthetic dataset generation
# Requirements:
# - 5000 rows
# - realistic ranges
# - correlations:
#   * night -> lower lighting
#   * high crime -> higher risk
# =========================================
def generate_synthetic_data(n_rows: int = 5000, random_state: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)

    hour_of_day = rng.integers(0, 24, size=n_rows)
    day_of_week = rng.integers(0, 7, size=n_rows)
    is_holiday = rng.binomial(1, 0.08, size=n_rows)

    crime_density = np.clip(rng.beta(2.0, 3.5, size=n_rows), 0, 1)
    crime_trend = np.clip(rng.normal(loc=0.0, scale=0.35, size=n_rows), -1, 1)
    crime_type_violence_ratio = np.clip(0.2 + 0.55 * crime_density + rng.normal(0, 0.12, size=n_rows), 0, 1)
    
    transit_lines_nearby = np.clip(np.round(rng.normal(8 - 2.5 * crime_density, 3.2, size=n_rows)), 0, 20).astype(int)
    transit_frequency = np.clip(transit_lines_nearby * rng.normal(2.5, 0.7, size=n_rows) + rng.normal(5, 4, size=n_rows), 0, 60)
    street_connectivity = np.clip(rng.beta(2.8, 2.2, size=n_rows), 0, 1)

    is_night = ((hour_of_day >= 20) | (hour_of_day <= 5)).astype(int)
    avg_lighting_quality = np.clip(78 - is_night * rng.normal(42, 10, size=n_rows) - 22 * crime_density + rng.normal(0, 8, size=n_rows), 0, 100)

    rush_hour = ((hour_of_day >= 7) & (hour_of_day <= 10)) | ((hour_of_day >= 17) & (hour_of_day <= 20))
    crowd_density = np.clip(25 + rush_hour.astype(int) * rng.normal(45, 12, size=n_rows) - is_night * rng.normal(18, 8, size=n_rows) + (transit_lines_nearby * 1.2) - (is_holiday * 8) + rng.normal(0, 10, size=n_rows), 0, 100)
    weather_visibility = np.clip(rng.normal(7.2, 2.0, size=n_rows), 0, 10)
    route_length_km = np.clip(rng.gamma(2.2, 3.0, size=n_rows), 0.5, 25)
    num_intersections = np.clip(np.round(route_length_km * rng.normal(2.2, 0.7, size=n_rows) + street_connectivity * 20 + rng.normal(0, 4, size=n_rows)), 0, 80).astype(int)

    # --- NEW: ARTIFICIAL INFLATION OF HIGH RISK ---
    # Force ~15% of the data to be inherently dangerous to fix the class imbalance
    high_risk_idx = rng.choice(n_rows, size=int(n_rows * 0.15), replace=False)
    crime_density[high_risk_idx] = rng.uniform(0.71, 1.0, len(high_risk_idx))
    avg_lighting_quality[high_risk_idx] = rng.uniform(0, 39, len(high_risk_idx))
    crowd_density[high_risk_idx] = rng.uniform(0, 29, len(high_risk_idx))
    hour_of_day[high_risk_idx] = rng.integers(22, 24, len(high_risk_idx))

    # --- NEW: PROBABILISTIC LABELING ---
    # Instead of a strict IF statement, we create a "Risk Score" and add noise. 
    # This simulates the unpredictability of the real world and gives the ML model a reason to exist.
    base_risk = (
        (crime_density * 0.4) + 
        ((100 - avg_lighting_quality) / 100 * 0.3) + 
        ((100 - crowd_density) / 100 * 0.2) + 
        (crime_type_violence_ratio * 0.1)
    )
    
    # Add random noise so it isn't perfectly predictable
    noisy_risk = base_risk + rng.normal(0, 0.1, size=n_rows)
    
    # Set the threshold so roughly 15-20% of data is labeled as 1
    threshold = np.percentile(noisy_risk, 80) 
    risk = (noisy_risk >= threshold).astype(int)

    df = pd.DataFrame({
        "crime_density": crime_density,
        "crime_trend": crime_trend,
        "crime_type_violence_ratio": crime_type_violence_ratio,
        "transit_lines_nearby": transit_lines_nearby,
        "transit_frequency": transit_frequency,
        "street_connectivity": street_connectivity,
        "avg_lighting_quality": avg_lighting_quality,
        "crowd_density": crowd_density,
        "weather_visibility": weather_visibility,
        "hour_of_day": hour_of_day,
        "day_of_week": day_of_week,
        "is_holiday": is_holiday,
        "route_length_km": route_length_km,
        "num_intersections": num_intersections,
        "risk": risk,
    })

    return df

In [5]:
# Cell 5: Build preprocessing transformer
# =========================================
def build_preprocessor(feature_columns):
    # All are numeric for now; scale for Logistic Regression stability
    numeric_features = feature_columns
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features)
        ],
        remainder="drop"
    )
    return preprocessor

In [6]:
# Cell 6: Generate, split, save artifacts
# =========================================
df = generate_synthetic_data(n_rows=CFG.n_rows, random_state=CFG.random_state)

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=CFG.random_state,
    stratify=y
)

preprocessor = build_preprocessor(FEATURE_COLUMNS)
preprocessor.fit(X_train)

# Save raw datasets
df.to_csv(f"{CFG.artifacts_dir}/synthetic_risk_dataset.csv", index=False)
X_train.to_csv(f"{CFG.artifacts_dir}/X_train.csv", index=False)
X_test.to_csv(f"{CFG.artifacts_dir}/X_test.csv", index=False)
y_train.to_csv(f"{CFG.artifacts_dir}/y_train.csv", index=False)
y_test.to_csv(f"{CFG.artifacts_dir}/y_test.csv", index=False)

# Save preprocessor + metadata
joblib.dump(preprocessor, f"{CFG.artifacts_dir}/preprocessor.joblib")
joblib.dump(FEATURE_COLUMNS, f"{CFG.artifacts_dir}/feature_columns.joblib")

print("Saved preprocessing artifacts to:", CFG.artifacts_dir)
print("Dataset shape:", df.shape)
print("Positive class ratio (risk=1):", df["risk"].mean().round(4))
df.head()

Saved preprocessing artifacts to: artifacts
Dataset shape: (5000, 15)
Positive class ratio (risk=1): 0.2


,crime_density,crime_trend,crime_type_violence_ratio,transit_lines_nearby,transit_frequency,street_connectivity,avg_lighting_quality,crowd_density,weather_visibility,hour_of_day,day_of_week,is_holiday,route_length_km,num_intersections,risk
0,0.037674,0.011464,0.292293,0,6.155043,0.118267,14.052424,9.920321,6.881024,2,1,0,3.654955,14,0
1,0.402278,0.329756,0.568138,6,26.549088,0.790343,74.201025,76.833424,8.729438,18,0,0,12.181684,50,0
2,0.743546,-0.397371,0.710440,5,14.805587,0.138602,58.299264,32.853224,6.914318,15,4,0,10.689194,18,0
3,0.396293,0.636399,0.253587,3,16.747879,0.481713,70.728990,67.778312,5.390805,10,4,0,3.207563,16,0
4,0.347783,-0.270229,0.377301,5,23.314224,0.755329,71.139105,67.836423,7.105967,10,5,0,3.062839,18,0


In [7]:
# Cell 7: Quick sanity checks for correlation requirements
# =========================================
night_mask = (df["hour_of_day"] >= 20) | (df["hour_of_day"] <= 5)
print("Avg lighting (night):", df.loc[night_mask, "avg_lighting_quality"].mean().round(2))
print("Avg lighting (day):", df.loc[~night_mask, "avg_lighting_quality"].mean().round(2))

high_crime_mask = df["crime_density"] > 0.7
print("Risk rate | high crime:", df.loc[high_crime_mask, "risk"].mean().round(4))
print("Risk rate | low crime :", df.loc[~high_crime_mask, "risk"].mean().round(4))

Avg lighting (night): 25.66
Avg lighting (day): 70.11
Risk rate | high crime: 0.7328
Risk rate | low crime : 0.0731
